# Decision Tree Weather Type Classification

This notebook builds one model only: a Decision Tree classifier for the Kaggle Weather Type Classification dataset. It includes preprocessing in a scikit-learn pipeline for numeric and categorical columns.

## 1. Imports

This cell imports the libraries used for data loading, preprocessing, model building, evaluation, and plotting.  
These tools are enough to take the dataset from raw CSV form all the way to final predictions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

## 2. Load Dataset

This cell reads the weather dataset from a CSV file.  
If the file name is correct and the CSV is in the same folder as the notebook, the dataset will load into a pandas DataFrame.

In [ ]:
DATA_PATH = "weather_classification_data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Dataset shape:", df.shape)
display(df.info())
df.describe(include="all")

## 3. Split Features and Target

This cell separates the input columns (**features**) from the output column (**target**).  
The target for this project is `Weather Type`, which is the label the model will learn to predict.

In [ ]:
TARGET_COLUMN = "Weather Type"

if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Expected target column '{TARGET_COLUMN}', but found: {list(df.columns)}")

X = df.drop(columns=[TARGET_COLUMN])

# Encode the target weather condition labels as numbers.
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[TARGET_COLUMN])
target_mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))

print("Feature columns:", list(X.columns))
print("Target encoding:", target_mapping)

In [ ]:
# First split: hold out 15% for the final test set.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Second split: take validation data from the remaining 85%.
# This gives approximately 70% training, 15% validation, and 15% testing.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,
    random_state=42,
    stratify=y_train_val
)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])
print("Testing rows:", X_test.shape[0])

## 4. Preprocessing and Decision Tree Pipeline

Numeric columns use median imputation for missing values. Categorical columns use most-frequent imputation and one-hot encoding. Decision Trees do not require feature scaling, so scaling is intentionally not included.

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
condition_features = [col for col in ["Cloud Cover", "Season", "Location"] if col in X.columns]

print("Numeric features:", numeric_features)
print("Categorical features encoded with OneHotEncoder:", categorical_features)
print("Condition features:", condition_features)

The next cell shows what condition encoding looks like. The model still uses the pipeline below so encoding is learned only from the training data.

In [ ]:
if condition_features:
    encoded_condition_preview = pd.get_dummies(
        X[condition_features],
        columns=condition_features,
        dtype=int
    )
    display(encoded_condition_preview.head())
else:
    print("No condition columns found for encoding.")

In [ ]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        criterion="gini",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        class_weight="balanced"
    ))
])

dt_pipeline

## 5. Train Model

In [ ]:
dt_pipeline.fit(X_train, y_train)

## 6. Validation Evaluation

In [ ]:
y_val_pred = dt_pipeline.predict(X_val)

validation_accuracy = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {validation_accuracy:.4f}")
print("\nValidation Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=target_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_val, y_val_pred, labels=range(len(target_encoder.classes_)))
cm_df = pd.DataFrame(cm, index=target_encoder.classes_, columns=target_encoder.classes_)
cm_df

## 7. Cross-Validation

In [ ]:
cv_scores = cross_val_score(dt_pipeline, X_train, y_train, cv=5, scoring="accuracy", n_jobs=1)

print("CV scores:", cv_scores)
print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"CV standard deviation: {cv_scores.std():.4f}")

## 8. Feature Importance

In [ ]:
feature_names = dt_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = dt_pipeline.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
importance_df.head(20).plot(
    kind="barh",
    x="feature",
    y="importance",
    figsize=(10, 7),
    legend=False,
    title="Top 20 Decision Tree Feature Importances"
)

## 9. Final Held-Out Test Set Evaluation

Use the test set once at the end after training and validation are complete.

In [ ]:
y_test_pred = dt_pipeline.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)
print(f"Final Test Accuracy: {test_accuracy:.4f}")
print("\nFinal Test Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=target_encoder.classes_))

In [ ]:
test_cm = confusion_matrix(y_test, y_test_pred, labels=range(len(target_encoder.classes_)))
test_cm_df = pd.DataFrame(test_cm, index=target_encoder.classes_, columns=target_encoder.classes_)
test_cm_df